# A First Lesson in Machine Learning
## Comprehensive Study Guide: Chapter 5 — Classification

---

### Table of Contents

- [5.1 The General Problem](#sec-51-general-problem)
- [5.2 Probabilistic Classifiers](#sec-52-probabilistic)
  - [5.2.1 The Bayes classifier](#sec-521-bayes-classifier)
    - [5.2.1.1 Likelihood — class-conditional distributions](#sec-5211-likelihood)
    - [5.2.1.2 Prior class distribution](#sec-5212-prior)
    - [5.2.1.3 Example — Gaussian class-conditionals](#sec-5213-gaussian-class-conditionals)
    - [5.2.1.4 Making predictions](#sec-5214-making-predictions)
    - [5.2.1.5 The naïve-Bayes assumption](#sec-5215-naive-bayes)
    - [5.2.1.6 Example — classifying text](#sec-5216-classifying-text)
    - [5.2.1.7 Smoothing](#sec-5217-smoothing)
  - [5.2.2 Logistic regression](#sec-522-logistic-regression)
    - [5.2.2.1 Motivation](#sec-5221-motivation)
    - [5.2.2.2 Non-linear decision functions](#sec-5222-nonlinear-decision)
    - [5.2.2.3 Non-parametric models — the Gaussian process](#sec-5223-gp)
- [5.3 Non-probabilistic Classifiers](#sec-53-nonprobabilistic)
  - [5.3.1 K-nearest neighbours](#sec-531-knn)
    - [5.3.1.1 Choosing K](#sec-5311-choosing-k)
  - [5.3.2 Support vector machines and other kernel methods](#sec-532-svm-kernel)
    - [5.3.2.1 The margin](#sec-5321-margin)
    - [5.3.2.2 Maximising the margin](#sec-5322-maximising-margin)
    - [5.3.2.3 Making predictions](#sec-5323-making-predictions)
    - [5.3.2.4 Support vectors](#sec-5324-support-vectors)
    - [5.3.2.5 Soft margins](#sec-5325-soft-margins)
    - [5.3.2.6 Kernels](#sec-5326-kernels)
  - [5.3.3 Summary](#sec-533-summary)
- [5.4 Assessing Classification Performance](#sec-54-assessing)
  - [5.4.1 Accuracy — 0/1 loss](#sec-541-accuracy-01)
  - [5.4.2 Sensitivity and specificity](#sec-542-sensitivity-specificity)
  - [5.4.3 The area under the ROC curve](#sec-543-auc-roc)
  - [5.4.4 Confusion matrices](#sec-544-confusion-matrices)
- [5.5 Discriminative and Generative Classifiers](#sec-55-disc-vs-gen)
- [5.6 Chapter Summary](#sec-56-chapter-summary)
- [5.7 Exercises](#sec-57-exercises)
- [5.8 Further Reading](#sec-58-further-reading)

---

Important Links

- Github Repo: https://github.com/omniV1/GCU_SWE_2023-2025/tree/main/AIT-104-Data-mining-machine-learning/Notes
- Google Colab: https://colab.research.google.com/

<a id="sec-51-general-problem"></a>

## 5.1 The General Problem

**Problem Setup:**

Typically, we are presented with a set of N training objects, x₁, …, xₙ. Each is a vector with dimension D. For each object we are also provided with a label tₙ that describes which class object n belongs to. This label typically takes an integer value.

**Classification Labels:**
- For two classes: tₙ = {0, 1} or tₙ = {−1, 1}
- More generally, for C classes: tₙ = {1, 2, …, C}

**Our Task:** Predict the class t_new for an unseen object x_new.

**Connection to Previous Chapters:**
The classification setup is very similar to regression (Chapters 1-3), where we had objects x₁, …, xₙ and associated real-valued labels (e.g., Olympic years and winning times). The key difference is that in classification, the response variable is an integer indicating a particular class rather than a real value.

**Real-World Applications:**
1. **Automatic disease diagnosis** - predicting whether a patient is healthy or unhealthy based on medical observations
2. **Text classification** - classifying documents into topics or as relevant/irrelevant for a particular user

**Key Challenges:**
- Handling uneven cost of making errors
- Handling complex data objects like text
- Managing uncertainty in predictions

[Back to TOC](#sec-51-general-problem)

<a id="sec-52-probabilistic"></a>

## 5.2 Probabilistic Classifiers

**Key Difference:** Probabilistic and non-probabilistic classifiers differ in the type of output they produce. In the probabilistic case, the output is the probability of a new object belonging to a particular class.

**Output Constraints:** For class c, the probability must satisfy:
- 0 ≤ P(T_new = c|x_new, X, t) ≤ 1
- ∑_c P(T_new = c|x_new, X, t) = 1

**Why Probabilities Matter:**
While we could just predict the most likely class, probabilities provide confidence levels. For example, in disease diagnosis:
- P(T_new = 1|x_new, X, t) = 0.6 vs P(T_new = 1|x_new, X, t) = 0.9
- Both suggest "diseased" classification, but 0.6 indicates much less certainty
- May require additional tests before making a decision

<a id="sec-521-bayes-classifier"></a>

### 5.2.1 The Bayes classifier

**Principle:** Choose class that maximizes posterior probability based on Bayes' rule.

**Goal:** Compute predictive probabilities for each of C potential classes. These probabilities form the basis of decision-making (assign to highest probability class) or expectation calculations.

**Bayes' Rule Application:**
From Bayes' rule, we can express the predictive probability:

P(T_new = c|x_new, X, t) = p(x_new|T_new = c, X, t) × P(T_new = c|X, t) / p(x_new|X, t)

**Key Components:**
1. **Likelihood:** p(x_new|T_new = c, X, t) - probability of x_new given it belongs to class c
2. **Prior:** P(T_new = c|X, t) - prior probability of class c
3. **Evidence:** p(x_new|X, t) - marginal likelihood (normalizing constant)

<a id="sec-5211-likelihood"></a>

#### 5.2.1.1 Likelihood — class-conditional distributions
The likelihood term p(x_new|T_new = c, X, t) represents the probability of observing x_new given that it belongs to class c. This is discussed in detail in Section 5.2.1.3 with the Gaussian example.

<a id="sec-5212-prior"></a>

#### 5.2.1.2 Prior class distribution
**Purpose:** P(T_new = c|X, t) enables us to specify prior beliefs about the class of x_new before observing it.

**Applications:**
- **Account for uneven class sizes** - handle rare vs common classes
- **Bias against rare classes** - set low prior probability, only classify as rare class if likelihood is very high
- **Prioritize rare class detection** - set high prior probability to catch rare instances (may increase false positives but reduce false negatives)

**Technical Constraints:**
- Must be positive: P(T_new = c|X, t) > 0
- Must sum to 1: ∑_c P(T_new = c|X, t) = 1

**Two Popular Choices:**
1. **Uniform prior:** P(T_new = c|X, t) = 1/C (equal probability for all classes)
2. **Class size prior:** P(T_new = c|X, t) = N_c/N (proportional to training set class sizes)
   - Where N = total training objects, N_c = objects in class c

<a id="sec-5213-gaussian-class-conditionals"></a>

#### 5.2.1.3 Example — Gaussian class-conditionals
**Setup:** Three-class dataset with two-dimensional attributes x_n = [x_{n1}, x_{n2}]^T and labels t_n ∈ {1, 2, 3}.

**Gaussian Class-Conditional Distribution:**
p(x_new|T_new = c) = (1/√(2π)^D |Σ_c|) × exp(-½(x_new - μ_c)^T Σ_c^{-1}(x_new - μ_c))

**Parameter Estimation:**
- Use training points X_c from class c
- Find μ_c and Σ_c that maximize likelihood of X_c
- This is a machine learning task: infer model parameters from data

**Bayesian Alternative:**
Instead of maximum likelihood, define prior p(μ_c, Σ_c) and compute posterior:
p(μ_c, Σ_c|X_c) ∝ p(X_c|μ_c, Σ_c) × p(μ_c, Σ_c)

Then compute likelihood by taking expectation:
p(x_new|T_new = c) = ∫ p(x_new|μ_c, Σ_c) × p(μ_c, Σ_c|X_c) dμ_c dΣ_c

**Advantage:** Bayesian approach provides uncertainty quantification in parameters.

<a id="sec-5214-making-predictions"></a>

#### 5.2.1.4 Making predictions
**Process:** Armed with class-conditional distributions and priors, we can compute posterior class probabilities for any new point x_new.

**Worked Example:** For x_new = [2, 0]^T, we compute:
1. **Likelihood values** for each class: p(x_new|T_new = c, μ_c, Σ_c)
2. **Prior probabilities:** P(T_new = c|X, t)
3. **Unnormalized posteriors:** likelihood × prior
4. **Normalized probabilities:** divide by sum of all unnormalized values

**Example Results:**
- Class 1: P(T_new = 1|x_new) ≈ 0.687 (most likely)
- Class 2: P(T_new = 2|x_new) ≈ 0.299 
- Class 3: P(T_new = 3|x_new) ≈ 0.015 (very unlikely)

**Classification Decision:** Assign x_new to class with highest posterior probability.

**Visualization:** By evaluating probabilities over a grid of points, we can draw classification probability contours showing decision boundaries.

**Potential Issues:** 
- High probabilities in regions with no training data (due to steep density decays)
- May not reflect true uncertainty in data-sparse regions
- Better approach: Use models that become less certain away from data (like binary response model from Chapter 4)

<a id="sec-5215-naive-bayes"></a>

#### 5.2.1.5 The naïve-Bayes assumption
**Problem with Full Gaussian Models:**
- High-dimensional Gaussians require many parameters (D means + D(D+1)/2 covariance terms)
- Computational complexity grows rapidly with dimensionality
- May overfit with limited data

**Naïve-Bayes Solution:**
**Assumption:** All attributes are conditionally independent given the class label.

**Mathematical Form:**
p(x_new|T_new = c) = ∏_{d=1}^D p(x_{new,d}|T_new = c)

**Advantages:**
- **Reduced parameters:** Only need D means and D variances per class (instead of full covariance matrix)
- **Computational efficiency:** Much faster training and prediction
- **Scalability:** Works well with high-dimensional data
- **Robustness:** Less prone to overfitting

**Trade-offs:**
- **Lost dependencies:** Cannot capture correlations between attributes
- **Approximation:** May not model true data distribution accurately
- **Performance:** Can still work well in practice despite independence assumption

**When to Use:**
- High-dimensional problems where full covariance estimation is impractical
- When computational efficiency is important
- When attribute independence is a reasonable approximation

<a id="sec-5216-classifying-text"></a>

#### 5.2.1.6 Example — classifying text
**Application:** Machine learning for automatic text classification is widely used due to the abundance of text data and difficulty of manually building classification rules.

**Dataset:** 20 newsgroups dataset - approximately 20,000 documents from 20 different newsgroups covering diverse topics (sport, computing, religion).

**Text Encoding Challenge:** Algorithms work with numerical data, so we need to encode documents as numerical vectors.

**Bag-of-Words Model:**
- **Vocabulary:** M unique words across all documents
- **Document representation:** Each document becomes an M-dimensional vector x_n
- **Word counts:** x_{nm} = number of times word m appears in document n
- **Ordering ignored:** "The quick brown fox" and "Fox brown quick the" have identical representations

**Naive Bayes for Text:**
Given large vocabularies, we use the naive-Bayes assumption:

p(x_new|T_new = c) = ∏_{m=1}^M p(x_{new,m}|T_new = c)

**Multinomial Class-Conditional Distribution:**
p(x_n|T_n = c, q_c) = (∑_{m=1}^M x_{nm})! / (∏_{m=1}^M x_{nm}!) × ∏_{m=1}^M q_{cm}^{x_{nm}}

Where q_c = [q_{c1}, ..., q_{cM}]^T are word probabilities for class c, with ∑_{m=1}^M q_{cm} = 1.

**Parameter Estimation:**
Maximum likelihood estimate for word probabilities in class c:
q_{cm} = ∑_{n∈c} x_{nm} / ∑_{n∈c} ∑_{m=1}^M x_{nm}

**Problem:** Zero probabilities for unseen words cause classification failures.

<a id="sec-5217-smoothing"></a>

#### 5.2.1.7 Smoothing
**The Zero Probability Problem:**
If word m never appears in class c training documents, then q_{cm} = 0. When a new document contains word m, the likelihood becomes zero, making P(T_new = c|x_new) = 0 regardless of other evidence.

**Solution - Dirichlet Prior:**
Place a Dirichlet prior on q_c to encode belief that all word probabilities are greater than 0:

p(q_c|α) = Γ(∑_{m=1}^M α_m) / (∏_{m=1}^M Γ(α_m)) × ∏_{m=1}^M q_{cm}^{α_m - 1}

**Simplified Form:** Assume α_m = α for all words.

**MAP Estimate with Smoothing:**
q_{cm} = (∑_{n∈c} x_{nm} + α - 1) / (∑_{n∈c} ∑_{m=1}^M x_{nm} + M(α - 1))

**Benefits:**
- **Prevents zeros:** For α > 1, all q_{cm} > 0
- **Regularization:** Larger α values push probabilities toward uniform distribution
- **Robustness:** Handles unseen words gracefully

**Results on 20 Newsgroups:**
- Training set: ≈11,000 documents
- Test set: ≈7,000 documents  
- **Accuracy: 78%** with α = 2 and uniform class priors
- **Error analysis:** Confusion between related topics (e.g., politics.guns vs politics.misc, religion.misc vs religion.christian)

**Key Insight:** Smoothing is essential for text classification and is another form of regularization that prevents overfitting to training data.

<a id="sec-522-logistic-regression"></a>

### 5.2.2 Logistic regression

**Connection to Chapter 4:** The binary response model from Chapter 4 is actually logistic regression, a binary classifier. Chapter 4 focused on the inference challenges, but here we discuss it from a classification perspective.

<a id="sec-5221-motivation"></a>

#### 5.2.2.1 Motivation
**Previous Motivation:** We wanted to use linear models (w^T x) but needed to transform output to probabilities [0,1].

**Formal Derivation - Log-Odds Ratio:**
The logistic likelihood is more formally derived by modeling the log-odds ratio:

log-odds = log(P(T_new = 1|x_new, w) / P(T_new = 0|x_new, w))

**Properties of Log-Odds:**
- **Unconstrained:** Can take any real value
- **Interpretation:** 
  - Large negative: P(T_new = 1) ≪ P(T_new = 0)
  - Large positive: P(T_new = 1) ≫ P(T_new = 0)

**Linear Model for Log-Odds:**
log-odds = w^T x_new

**Deriving Logistic Function:**
Rearranging and using P(T_new = 0) = 1 - P(T_new = 1):

P(T_new = 1|x_new, w) = 1 / (1 + exp(-w^T x_new)) = σ(w^T x_new)

**Key Insight:** By using logistic likelihood, we're actually modeling log-odds with a linear model. This is part of the **generalized linear models** family - linear models transformed to model quantities of interest.

<a id="sec-5222-nonlinear-decision"></a>

#### 5.2.2.2 Non-linear decision functions
**Linear Boundaries:** Individual w values in Chapter 4 produced straight decision boundaries. Curved boundaries only appeared when averaging over multiple w values (Laplace/MH).

**Non-linear Extension:** Expand x to include higher-order terms (similar to polynomial regression in Chapter 1).

**Example Model:**
For x = [x₁, x₂]^T, use:
log-odds = w₀ + w₁x₁ + w₂x₂ + w₃x₁² + w₄x₂² + w₅x₁x₂

**Implementation:**
1. **Feature expansion:** Create augmented feature vector with squared and interaction terms
2. **MAP estimation:** Find optimal w with Gaussian prior
3. **Probability computation:** P(T_new = 1|x_new) = σ(w^T φ(x_new)) where φ(x) is the expanded feature vector

**Results:** Non-linear decision boundaries that can handle complex classification patterns.

**Cautions:**
- **Overfitting risk:** Same complexity issues as polynomial regression
- **Generalization:** More complex models may perform worse on new data
- **Parameter uncertainty:** Non-linear boundaries in Laplace/MH come from averaging, not single w values

<a id="sec-5223-gp"></a>

#### 5.2.2.3 Non-parametric models — the Gaussian process
**Limitation of Parametric Models:** Throughout the book, we've used models of form w^T x with fixed parameterizations. This restricts us to specific function families.

**Gaussian Process Alternative:** 
- **Non-parametric:** Model complexity grows with data
- **Flexible:** Can represent any function in the limit
- **Uncertainty:** Provides natural uncertainty quantification

**Concept:** Instead of learning fixed parameters w, we place a prior directly over functions f(x) that maps inputs to log-odds.

**Advantages:**
- **Automatic complexity:** Model adapts its complexity to the data
- **No feature engineering:** Learns appropriate transformations automatically  
- **Uncertainty:** Provides predictive uncertainty naturally

**Trade-offs:**
- **Computational cost:** O(N³) complexity for N training points
- **Scalability:** Becomes impractical for very large datasets
- **Interpretability:** Less interpretable than linear models

[Back to TOC](#sec-51-general-problem)

<a id="sec-53-nonprobabilistic"></a>

## 5.3 Non-probabilistic Classifiers

**Key Difference:** Unlike probabilistic classifiers, these produce direct class assignments t_new = c rather than probabilities P(T_new = c|x_new, X, t).

**Coverage:** We examine two popular algorithms:
1. **K-nearest neighbours (KNN)** - Simple, effective, no training phase
2. **Support Vector Machine (SVM)** - Excellent performance, introduces kernel methods

<a id="sec-531-knn"></a>

### 5.3.1 K-nearest neighbours

**Why KNN is Popular:**
- **Simplicity:** Easy to understand and implement
- **Excellent empirical performance:** Often works very well in practice
- **Versatility:** Handles binary and multiclass problems
- **Flexibility:** No assumptions about decision boundary shape
- **No training phase:** Algorithm is the classification process itself

**Classification Process:**
1. **Given:** N training objects {x_n, t_n} and new object x_new
2. **Find:** K training points closest to x_new (using chosen distance measure)
3. **Assign:** t_new = majority class among K neighbors

**Example:** Figure 5.8 shows K=3 classification:
- Test point A: 2 squares + 1 circle → classified as square
- Test point B: 3 circles → classified as circle

**Handling Ties:**
- **Problem:** Equal votes from multiple classes
- **Solutions:**
  - Random assignment (not ideal - inconsistent results)
  - Use odd K for binary classification
  - **Weighted voting:** Closer neighbors have more influence

**Distance Flexibility:**
- **Any distance measure** can be used (Euclidean, Manhattan, etc.)
- **Works with any data type** where distance is defined:
  - Strings (edit distance)
  - Graphs (graph edit distance)
  - Images (pixel-based distances)

<a id="sec-5311-choosing-k"></a>

#### 5.3.1.1 Choosing K
**The K Selection Problem:** K is the only hyperparameter that needs tuning.

**K Too Small (e.g., K=1):**
- **Overfitting:** Decision boundary becomes very complex
- **Noise sensitivity:** Single noisy/mislabeled points create "islands"
- **Example:** Figure 5.9(a) shows three noise-induced islands that misclassify large regions

**K Too Large:**
- **Underfitting:** Decision boundary becomes too smooth
- **Class imbalance issues:** Minority class can be completely overwhelmed
- **Example:** With N₀=50, N₁=10, if K≥21, no point can ever be classified as class 1

**Optimal K Range:**
- **Sweet spot:** Balance between overfitting and underfitting
- **Cross-validation:** Use k-fold CV to find best K empirically
- **Figure 5.9(b):** K=5 provides good regularization without losing true patterns

**Practical Guidelines:**
- **Start with K=5 or K=7** for most problems
- **Use odd K** for binary classification to avoid ties
- **Consider class imbalance:** Larger K may favor majority class
- **Cross-validate:** Let data determine optimal K through systematic search

**Regularization Effect:** Increasing K acts as regularization, smoothing the decision boundary and reducing overfitting risk.

<a id="sec-532-svm-kernel"></a>

### 5.3.2 Support vector machines and other kernel methods

**Why SVMs are Popular:**
- **Excellent empirical performance:** Hard to beat for many applications
- **High-dimensional data:** Particularly effective when attributes >> training objects
- **Parameter efficiency:** Number of parameters relates to training objects, not attributes
- **Kernel methods:** Introduction to powerful non-linear techniques

**Basic SVM Setup:**
- **Binary classifier:** Uses class labels {+1, -1}
- **Linear decision boundary:** w^T x_new + b
- **Classification rule:** t_new = sign(w^T x_new + b)
- **Learning task:** Find w and b by maximizing the margin

<a id="sec-5321-margin"></a>

#### 5.3.2.1 The margin
**Definition:** The margin γ is the perpendicular distance from the decision boundary to the closest points on either side.

**Why Maximize Margin:**
- **Intuition:** Larger margin = more robust decision boundary
- **Example:** Figure 5.12(a) shows larger margin (more sensible boundary) vs Figure 5.12(b) with smaller margin (poor boundary)
- **Robustness:** Larger margin provides better generalization to new data

**Margin Computation:**
Using closest points x₁ and x₂ from each class:
- Vector joining points: x₁ - x₂
- Direction perpendicular to boundary: w/||w||
- Margin: γ = (1/2||w||) × (x₁ - x₂)^T × (w/||w||)

<a id="sec-5322-maximising-margin"></a>

#### 5.3.2.2 Maximising the margin
**Scaling Invariance:** The decision function sign(w^T x + b) is invariant to scaling by positive constants.

**Convenient Scaling:** Fix scaling so that for closest points:
- Class +1: w^T x + b = +1
- Class -1: w^T x + b = -1

**Simplified Margin:** With this scaling, γ = 1/||w||

**Optimization Problem:** Maximize γ = 1/||w|| subject to constraints:
- For all class +1 points: w^T x_n + b ≥ 1
- For all class -1 points: w^T x_n + b ≤ -1

**Unified Constraints:** Using t_n ∈ {+1, -1}:
t_n(w^T x_n + b) ≥ 1 for all n

**Quadratic Programming Form:** Minimize (1/2)||w||² subject to t_n(w^T x_n + b) ≥ 1

**Lagrange Multipliers:** Convert to unconstrained optimization:
L = (1/2)||w||² - ∑_n α_n[t_n(w^T x_n + b) - 1]

**Optimality Conditions:**
- ∂L/∂w = 0 → w = ∑_n α_n t_n x_n
- ∂L/∂b = 0 → ∑_n α_n t_n = 0

**Dual Problem:** Maximize ∑_n α_n - (1/2)∑_m ∑_n α_m α_n t_m t_n x_m^T x_n
Subject to: α_n ≥ 0 and ∑_n α_n t_n = 0

<a id="sec-5323-making-predictions"></a>

#### 5.3.2.3 Making predictions
**Decision Function with Dual Variables:**
Substituting w = ∑_n α_n t_n x_n into the decision function:
t_new = sign(∑_n α_n t_n x_n^T x_new + b)

**Computing b:**
For support vectors (closest points): t_n(w^T x_n + b) = 1
Therefore: b = t_n - w^T x_n for any support vector x_n

**Complete Prediction Process:**
1. Compute w^T x_new = ∑_n α_n t_n x_n^T x_new
2. Add bias: w^T x_new + b
3. Apply sign function: t_new = sign(w^T x_new + b)

<a id="sec-5324-support-vectors"></a>

#### 5.3.2.4 Support vectors
**Definition:** Support vectors are the training points closest to the maximum margin decision boundary.

**Key Properties:**
- **Define the boundary:** The decision boundary depends only on support vectors
- **Sparse solution:** Only support vectors have α_n > 0
- **Efficiency:** Can discard all non-support vectors without changing the boundary

**Computational Advantage:**
- **KNN:** Must compute distances to all training points
- **SVM:** Only uses support vectors for classification
- **Scalability:** Particularly beneficial for large datasets

**Example:** Figure 5.14 shows only 3 support vectors out of many training points.

**Potential Problem:** Individual support vectors can have excessive influence if they're outliers (Figure 5.15).

<a id="sec-5325-soft-margins"></a>

#### 5.3.2.5 Soft margins
**Problem with Hard Margin:** All training points must be correctly classified, leading to overfitting when data contains noise or outliers.

**Soft Margin Solution:** Allow some training points to violate the margin constraints.

**Modified Constraints:** Introduce slack variables ξ_n ≥ 0:
t_n(w^T x_n + b) ≥ 1 - ξ_n

**Interpretation of ξ_n:**
- ξ_n = 0: Point is correctly classified with margin ≥ 1
- 0 < ξ_n ≤ 1: Point is correctly classified but within margin
- ξ_n > 1: Point is misclassified

**Modified Optimization:** Minimize (1/2)||w||² + C∑_n ξ_n
Subject to: t_n(w^T x_n + b) ≥ 1 - ξ_n and ξ_n ≥ 0

**Parameter C:** Controls trade-off between margin maximization and constraint violations.
- **Large C:** Few violations allowed (closer to hard margin)
- **Small C:** More violations allowed (softer margin)

**Dual Form:** Same as hard margin but with upper bound: 0 ≤ α_n ≤ C

**Effect:** Limits the influence of any single training point, reducing overfitting.

<a id="sec-5326-kernels"></a>

#### 5.3.2.6 Kernels
**Limitation of Linear Boundaries:** Some datasets (like Figure 5.17) cannot be separated by straight lines.

**Kernel Trick:** Transform data to higher-dimensional space where linear separation is possible.

**Key Insight:** SVM optimization and decision functions only involve inner products x_n^T x_m, never x_n alone.

**Kernel Functions:** Replace inner products with kernel functions k(x_n, x_m) = φ(x_n)^T φ(x_m)

**Advantages:**
- **No explicit transformation:** Never compute φ(x_n) explicitly
- **High-dimensional spaces:** Can use infinite-dimensional feature spaces
- **Flexibility:** Any valid kernel function can be used

**Common Kernels:**
- **Linear:** k(x_n, x_m) = x_n^T x_m
- **Polynomial:** k(x_n, x_m) = (x_n^T x_m + c)^d
- **RBF/Gaussian:** k(x_n, x_m) = exp(-γ||x_n - x_m||²)

**Example:** Figure 5.18 shows successful classification of Figure 5.17 data using Gaussian kernel.

**Kernel Methods:** SVMs introduced the concept of kernel methods, which extend beyond classification to regression, clustering, and other machine learning tasks.

<a id="sec-533-summary"></a>

### 5.3.3 Summary

**Non-probabilistic Classifiers Overview:**

**KNN Advantages:**
- Simple to understand and implement
- No training phase required
- Excellent empirical performance
- Works with any distance measure
- Handles both binary and multiclass problems

**KNN Considerations:**
- Choice of K is critical (balance overfitting vs underfitting)
- Can be sensitive to irrelevant features
- Computationally expensive for large datasets

**SVM Advantages:**
- Excellent empirical performance across many domains
- Effective with high-dimensional data
- Sparse solution (only support vectors matter)
- Kernel methods enable non-linear boundaries
- Strong theoretical foundation

**SVM Considerations:**
- Binary classification only (though multiclass extensions exist)
- Requires parameter tuning (C for soft margin, kernel parameters)
- Less interpretable than linear models with kernels

[Back to TOC](#sec-51-general-problem)

<a id="sec-54-assessing"></a>

## 5.4 Assessing Classification Performance

**Setup:** Assume we have N test examples with known true labels t₁*, ..., t_N* and classifier predictions t₁, ..., t_N.

<a id="sec-541-accuracy-01"></a>

### 5.4.1 Accuracy — 0/1 loss
**Definition:** Raw classification accuracy, also known as 0/1 loss.

**Calculation:** For each test point, loss = 0 if prediction is correct, 1 if incorrect. Average over all N test points gives the error rate.

**Interpretation:** Proportion of test objects incorrectly classified. Lower values are better.

**Limitations:**
- **Context dependent:** Is 0.2 error rate good or bad?
- **Class imbalance problems:** In imbalanced datasets, high accuracy can be misleading
- **Example:** If 80% of data is class 1, always predicting class 1 gives 20% error rate

**When to Use:** Only when classes are roughly balanced and you need a simple performance measure.

<a id="sec-542-sensitivity-specificity"></a>

### 5.4.2 Sensitivity and specificity
**Context:** Binary classification (e.g., disease detection: t = 0 healthy, t = 1 diseased).

**Four Key Quantities:**
- **True Positives (TP):** Objects with t* = 1 classified as t = 1 (diseased diagnosed as diseased)
- **True Negatives (TN):** Objects with t* = 0 classified as t = 0 (healthy diagnosed as healthy)
- **False Positives (FP):** Objects with t* = 0 classified as t = 1 (healthy diagnosed as diseased)
- **False Negatives (FN):** Objects with t* = 1 classified as t = 0 (diseased diagnosed as healthy)

**Sensitivity (Recall):** Se = TP / (TP + FN)
- Proportion of diseased people correctly identified as diseased
- Measures ability to detect positive cases

**Specificity:** Sp = TN / (TN + FP)
- Proportion of healthy people correctly identified as healthy
- Measures ability to detect negative cases

**Interpretation:**
- Both range from 0 to 1
- Ideal: Se = Sp = 1 (perfect detection)
- **Trade-offs:** Often must balance sensitivity vs specificity based on application needs
- **Rare disease example:** May prioritize high sensitivity (catch all diseased) over specificity (accept some false positives)

<a id="sec-543-auc-roc"></a>

### 5.4.3 The area under the ROC curve
**Motivation:** Many classifiers provide real-valued outputs (probabilities, SVM scores) that are thresholded for classification.

**ROC Curve:** Plots sensitivity vs (1 - specificity) = false positive rate across different threshold values.

**Curve Properties:**
- **Start:** (0, 0) - threshold that never classifies as positive
- **End:** (1, 1) - threshold that never classifies as negative
- **Perfect classifier:** Curve goes straight to top-left corner (Se = 1, 1-Sp = 0)
- **Random classifier:** Straight line from (0, 0) to (1, 1)

**Area Under Curve (AUC):**
- **Perfect classifier:** AUC = 1.0
- **Random classifier:** AUC = 0.5
- **Good classifier:** AUC closer to 1.0

**Advantages over 0/1 Loss:**
- **Class imbalance robust:** Uses sensitivity/specificity which handle imbalance better
- **Threshold independent:** Evaluates performance across all possible thresholds
- **Single metric:** Combines sensitivity and specificity into one value

**Limitations:**
- **Binary only:** Doesn't generalize to multiclass problems
- **Multiclass workaround:** Analyze as multiple binary problems (one-vs-rest)

<a id="sec-544-confusion-matrices"></a>

### 5.4.4 Confusion matrices
**Purpose:** Detailed breakdown of classification results, especially useful for multiclass problems.

**Binary Confusion Matrix:**
```
                 True Class
               1       0
Predicted 1   TP      FP
         0   FN      TN
```

**Multiclass Confusion Matrix:**
- Rows: Predicted classes
- Columns: True classes
- Diagonal: Correct classifications
- Off-diagonal: Misclassifications

**Benefits:**
- **Error analysis:** Identify which classes are confused with each other
- **Performance insights:** Understand failure modes of the classifier
- **Improvement guidance:** Focus on problematic class pairs
- **Multiclass friendly:** Works naturally with any number of classes

**Example from 20 Newsgroups:**
- Classes 19 and 17 often confused (both politics-related)
- Classes 20 and 16 often confused (both religion-related)
- Suggests combining similar classes or collecting more data for problematic classes

[Back to TOC](#sec-51-general-problem)

<a id="sec-55-disc-vs-gen"></a>

## 5.5 Discriminative and Generative Classifiers

**Alternative Classification Taxonomy:** Beyond probabilistic vs non-probabilistic, classifiers can be categorized as generative or discriminative.

**Generative Classifiers:**
- **Approach:** Define a model for each class, then assign new objects to the model that fits best
- **Examples:** Bayesian classifier (Section 5.2.1)
- **Philosophy:** Model the data generation process for each class
- **Advantages:** Can generate new samples, naturally handle missing data
- **Disadvantages:** May require more data, can be computationally expensive

**Discriminative Classifiers:**
- **Approach:** Explicitly define decision boundaries between classes
- **Examples:** SVM (Section 5.3.2), Logistic regression (Section 5.2.2)
- **Philosophy:** Focus on the decision boundary rather than data generation
- **Advantages:** Often more efficient, can work well with less data
- **Disadvantages:** Cannot generate samples, harder to handle missing data

**Key Insight:** Both approaches can be effective, but they solve the classification problem from different perspectives. The choice often depends on the specific problem requirements and available data.

[Back to TOC](#sec-51-general-problem)

<a id="sec-56-chapter-summary"></a>

## 5.6 Chapter Summary

**Four Classification Algorithms Covered:**

**Probabilistic Classifiers:**
1. **Bayesian Classifier:** Uses Bayes' rule with class-conditional distributions and priors
   - Gaussian class-conditionals for continuous data
   - Naive Bayes assumption for high-dimensional problems
   - Text classification with multinomial distributions and smoothing
2. **Logistic Regression:** Models log-odds with linear functions
   - Can be extended to non-linear boundaries through feature expansion
   - Part of generalized linear models family

**Non-probabilistic Classifiers:**
3. **K-nearest Neighbors:** Simple, instance-based learning
   - No training phase, choice of K is critical
   - Works with any distance measure, handles multiclass naturally
4. **Support Vector Machine:** Maximizes margin for robust classification
   - Sparse solution using only support vectors
   - Soft margins handle noise and outliers
   - Kernel methods enable non-linear boundaries

**Performance Evaluation:**
- **Accuracy/0/1 Loss:** Simple but problematic with class imbalance
- **Sensitivity/Specificity:** Better for binary classification, handles imbalance
- **ROC/AUC:** Threshold-independent evaluation, combines sensitivity/specificity
- **Confusion Matrices:** Detailed error analysis, works with multiclass problems

**Key Takeaways:**
- Classification algorithms vary in complexity, assumptions, and output types
- Performance evaluation must consider class imbalance and application requirements
- No single algorithm dominates all problems - choice depends on data and requirements
- Understanding trade-offs helps select appropriate methods for specific applications

[Back to TOC](#sec-51-general-problem)

<a id="sec-57-exercises"></a>

## 5.7 Exercises

**5.1** Assuming Σ_c = I for all classes, compute the posterior density p(μ_c|X_c) for the parameter μ_c of a Bayesian classifier where the set of training objects in class c is given by x₁, ..., x_{N_c}. Assume a Gaussian prior on p(μ_c).

**5.2** Using the posterior computed in the previous exercise, compute the expected likelihood:
E[p(x_new|μ_c, Σ_c)] = ∫ p(x_new|μ_c, Σ_c) × p(μ_c|X_c) dμ_c

**5.3** Compute the maximum likelihood estimates of μ_c and Σ_c for class c of a Bayesian classifier with Gaussian class-conditionals and a set of N_c objects belonging to class c, x₁, ..., x_{N_c}.

**5.4** Compute the maximum likelihood estimates of q_{mc} for class c of a Bayesian classifier with multinomial class-conditionals and a set of N_c, M-dimensional objects belonging to class c, x₁, ..., x_{N_c}.

**5.5** For a Bayesian classifier with multinomial class-conditionals with M-dimensional parameters q_c, compute the posterior Dirichlet for class c when the prior over q_c is a Dirichlet with constant parameter α and the observations belonging to class c are the N_c observations x₁, ..., x_{N_c}.

**5.6** Compute the expected multinomial likelihood using the posterior from Exercise 5.5.

*Note: Worked solutions would typically be provided in a separate solutions manual.*

[Back to TOC](#sec-51-general-problem)

<a id="sec-58-further-reading"></a>

## 5.8 Further Reading

**Recommended Texts for Deeper Understanding:**

**General Classification:**
- Hastie, T., Tibshirani, R., & Friedman, J. (2009). The Elements of Statistical Learning. Springer.
- Bishop, C. M. (2006). Pattern Recognition and Machine Learning. Springer.

**Support Vector Machines:**
- Cristianini, N., & Shawe-Taylor, J. (2000). An Introduction to Support Vector Machines. Cambridge University Press.
- Schölkopf, B., & Smola, A. J. (2002). Learning with Kernels. MIT Press.

**Bayesian Methods:**
- Murphy, K. P. (2012). Machine Learning: A Probabilistic Perspective. MIT Press.
- Gelman, A., et al. (2013). Bayesian Data Analysis. CRC Press.

**Performance Evaluation:**
- Powers, D. M. W. (2011). Evaluation: From Precision, Recall and F-Measure to ROC, Informedness, Markedness & Correlation.
- Fawcett, T. (2006). An Introduction to ROC Analysis. Pattern Recognition Letters, 27(8), 861-874.

**Online Resources:**
- Scikit-learn documentation: https://scikit-learn.org/
- MATLAB Statistics and Machine Learning Toolbox documentation

[Back to TOC](#sec-51-general-problem)


In [ ]:
# Import necessary libraries
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_classification, make_circles
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import GaussianNB, MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, roc_auc_score, roc_curve
from sklearn.preprocessing import StandardScaler
import seaborn as sns
from scipy.stats import multivariate_normal
import warnings
warnings.filterwarnings('ignore')

# Set random seed for reproducibility
np.random.seed(42)

print("=== Classification Algorithms: Coding Examples ===\n")


## Code Examples: Classification Algorithms

This section provides practical implementations of the four main classification algorithms covered in this chapter. Each example includes data generation, model training, prediction, and evaluation.


In [ ]:
# 1. Generate synthetic datasets for demonstration
print("1. Generating synthetic datasets...")

# Dataset 1: Simple 2D classification (for visualization)
X_simple, y_simple = make_classification(
    n_samples=300, n_features=2, n_redundant=0, n_informative=2,
    n_clusters_per_class=1, random_state=42
)

# Dataset 2: Circular classification (for non-linear boundaries)
X_circles, y_circles = make_circles(n_samples=300, noise=0.1, factor=0.3, random_state=42)

# Dataset 3: High-dimensional classification
X_highdim, y_highdim = make_classification(
    n_samples=1000, n_features=20, n_informative=15, n_redundant=5,
    n_clusters_per_class=1, random_state=42
)

# Split datasets
X_simple_train, X_simple_test, y_simple_train, y_simple_test = train_test_split(
    X_simple, y_simple, test_size=0.3, random_state=42
)

X_circles_train, X_circles_test, y_circles_train, y_circles_test = train_test_split(
    X_circles, y_circles, test_size=0.3, random_state=42
)

X_highdim_train, X_highdim_test, y_highdim_train, y_highdim_test = train_test_split(
    X_highdim, y_highdim, test_size=0.3, random_state=42
)

print(f"   Simple dataset: {X_simple_train.shape[0]} train, {X_simple_test.shape[0]} test")
print(f"   Circles dataset: {X_circles_train.shape[0]} train, {X_circles_test.shape[0]} test")
print(f"   High-dim dataset: {X_highdim_train.shape[0]} train, {X_highdim_test.shape[0]} test\n")


### Example 1: Naive Bayes Classifier (Gaussian)

**Implementation:** Manual Gaussian Naive Bayes following the mathematical formulation from Section 5.2.1.


In [ ]:
class GaussianNaiveBayes:
    """
    Manual implementation of Gaussian Naive Bayes classifier
    Following the mathematical formulation from Section 5.2.1.5
    """
    
    def __init__(self):
        self.class_means = {}
        self.class_stds = {}
        self.class_priors = {}
        self.classes = None
    
    def fit(self, X, y):
        """
        Fit the Gaussian Naive Bayes model
        Implements the Naive Bayes assumption: features are independent given class
        """
        self.classes = np.unique(y)
        n_classes = len(self.classes)
        n_samples, n_features = X.shape
        
        # Calculate class priors (Section 5.2.1.2)
        for c in self.classes:
            class_mask = (y == c)
            self.class_priors[c] = np.sum(class_mask) / n_samples
            
            # Calculate means and standard deviations for each feature (Naive Bayes assumption)
            X_class = X[class_mask]
            self.class_means[c] = np.mean(X_class, axis=0)
            self.class_stds[c] = np.std(X_class, axis=0, ddof=1)
            
            # Add small epsilon to avoid division by zero
            self.class_stds[c] = np.maximum(self.class_stds[c], 1e-9)
    
    def _calculate_likelihood(self, x, mean, std):
        """
        Calculate likelihood using Gaussian distribution
        p(x|class) = ∏_d p(x_d|class) (Naive Bayes assumption)
        """
        # Gaussian likelihood for each feature
        likelihoods = (1 / (np.sqrt(2 * np.pi) * std)) * np.exp(-0.5 * ((x - mean) / std) ** 2)
        # Product over features (Naive Bayes assumption)
        return np.prod(likelihoods)
    
    def predict_proba(self, X):
        """
        Predict class probabilities using Bayes' rule
        P(class|x) = P(x|class) * P(class) / P(x)
        """
        n_samples = X.shape[0]
        probabilities = np.zeros((n_samples, len(self.classes)))
        
        for i, x in enumerate(X):
            class_probs = []
            
            for c in self.classes:
                # Likelihood: P(x|class)
                likelihood = self._calculate_likelihood(x, self.class_means[c], self.class_stds[c])
                
                # Posterior = Likelihood * Prior
                posterior = likelihood * self.class_priors[c]
                class_probs.append(posterior)
            
            # Normalize probabilities (divide by evidence)
            class_probs = np.array(class_probs)
            probabilities[i] = class_probs / np.sum(class_probs)
        
        return probabilities
    
    def predict(self, X):
        """Predict class labels"""
        probabilities = self.predict_proba(X)
        return self.classes[np.argmax(probabilities, axis=1)]

# Test the manual implementation
print("2. Testing Manual Gaussian Naive Bayes...")
manual_nb = GaussianNaiveBayes()
manual_nb.fit(X_simple_train, y_simple_train)

# Compare with sklearn implementation
sklearn_nb = GaussianNB()
sklearn_nb.fit(X_simple_train, y_simple_train)

# Make predictions
manual_pred = manual_nb.predict(X_simple_test)
sklearn_pred = sklearn_nb.predict(X_simple_test)

# Compare accuracies
manual_acc = accuracy_score(y_simple_test, manual_pred)
sklearn_acc = accuracy_score(y_simple_test, sklearn_pred)

print(f"   Manual Naive Bayes accuracy: {manual_acc:.4f}")
print(f"   Sklearn Naive Bayes accuracy: {sklearn_acc:.4f}")
print(f"   Predictions match: {np.array_equal(manual_pred, sklearn_pred)}\n")


### Example 2: Logistic Regression with Feature Expansion

**Implementation:** Logistic regression with polynomial feature expansion for non-linear decision boundaries (Section 5.2.2.2).


In [ ]:
def create_polynomial_features(X, degree=2):
    """
    Create polynomial features for non-linear decision boundaries
    Following Section 5.2.2.2: Non-linear decision functions
    """
    n_samples, n_features = X.shape
    
    # Start with original features
    X_poly = X.copy()
    
    # Add polynomial terms
    for d in range(2, degree + 1):
        for i in range(n_features):
            X_poly = np.column_stack([X_poly, X[:, i] ** d])
    
    # Add interaction terms for degree >= 2
    if degree >= 2:
        for i in range(n_features):
            for j in range(i + 1, n_features):
                X_poly = np.column_stack([X_poly, X[:, i] * X[:, j]])
    
    return X_poly

# Test logistic regression with different feature expansions
print("3. Testing Logistic Regression with Feature Expansion...")

# Linear logistic regression
lr_linear = LogisticRegression(random_state=42, max_iter=1000)
lr_linear.fit(X_circles_train, y_circles_train)
lr_linear_pred = lr_linear.predict(X_circles_test)
lr_linear_acc = accuracy_score(y_circles_test, lr_linear_pred)

# Logistic regression with polynomial features
X_circles_train_poly = create_polynomial_features(X_circles_train, degree=2)
X_circles_test_poly = create_polynomial_features(X_circles_test, degree=2)

lr_poly = LogisticRegression(random_state=42, max_iter=1000)
lr_poly.fit(X_circles_train_poly, y_circles_train)
lr_poly_pred = lr_poly.predict(X_circles_test_poly)
lr_poly_acc = accuracy_score(y_circles_test, lr_poly_pred)

print(f"   Linear Logistic Regression accuracy: {lr_linear_acc:.4f}")
print(f"   Polynomial Logistic Regression accuracy: {lr_poly_acc:.4f}")
print(f"   Feature expansion improvement: {lr_poly_acc - lr_linear_acc:.4f}\n")

# Demonstrate log-odds interpretation (Section 5.2.2.1)
print("   Log-odds interpretation:")
lr_linear_proba = lr_linear.predict_proba(X_circles_test[:5])
for i, prob in enumerate(lr_linear_proba):
    class_1_prob = prob[1]
    log_odds = np.log(class_1_prob / (1 - class_1_prob))
    print(f"   Sample {i+1}: P(class=1) = {class_1_prob:.3f}, log-odds = {log_odds:.3f}")
print()


### Example 3: K-Nearest Neighbors with K Selection

**Implementation:** KNN with cross-validation for optimal K selection (Section 5.3.1.1).


In [ ]:
from sklearn.model_selection import cross_val_score

def find_optimal_k(X_train, y_train, max_k=20, cv_folds=5):
    """
    Find optimal K using cross-validation
    Following Section 5.3.1.1: Choosing K
    """
    k_values = range(1, max_k + 1, 2)  # Only odd numbers to avoid ties
    cv_scores = []
    
    for k in k_values:
        knn = KNeighborsClassifier(n_neighbors=k)
        scores = cross_val_score(knn, X_train, y_train, cv=cv_folds, scoring='accuracy')
        cv_scores.append(scores.mean())
    
    # Find K with highest cross-validation score
    optimal_k = k_values[np.argmax(cv_scores)]
    best_score = max(cv_scores)
    
    return optimal_k, best_score, dict(zip(k_values, cv_scores))

# Find optimal K for different datasets
print("4. Finding Optimal K for KNN...")

# For simple dataset
optimal_k_simple, best_score_simple, k_scores_simple = find_optimal_k(
    X_simple_train, y_simple_train, max_k=15
)

# For circles dataset
optimal_k_circles, best_score_circles, k_scores_circles = find_optimal_k(
    X_circles_train, y_circles_train, max_k=15
)

print(f"   Simple dataset - Optimal K: {optimal_k_simple}, CV Score: {best_score_simple:.4f}")
print(f"   Circles dataset - Optimal K: {optimal_k_circles}, CV Score: {best_score_circles:.4f}")

# Demonstrate the effect of different K values
print("\n   Effect of K on performance:")
for k in [1, 3, 5, 7, 15]:
    knn = KNeighborsClassifier(n_neighbors=k)
    knn.fit(X_simple_train, y_simple_train)
    pred = knn.predict(X_simple_test)
    acc = accuracy_score(y_simple_test, pred)
    print(f"   K={k}: Test accuracy = {acc:.4f}")

print()

# Test KNN on test set with optimal K
knn_optimal = KNeighborsClassifier(n_neighbors=optimal_k_simple)
knn_optimal.fit(X_simple_train, y_simple_train)
knn_pred = knn_optimal.predict(X_simple_test)
knn_acc = accuracy_score(y_simple_test, knn_pred)

print(f"   Final KNN accuracy (K={optimal_k_simple}): {knn_acc:.4f}\n")


### Example 4: Support Vector Machine with Kernels

**Implementation:** SVM with different kernels and parameter tuning (Section 5.3.2).


In [ ]:
from sklearn.model_selection import GridSearchCV

def compare_svm_kernels(X_train, X_test, y_train, y_test):
    """
    Compare different SVM kernels and parameters
    Following Section 5.3.2: Support Vector Machines and other kernel methods
    """
    results = {}
    
    # Linear SVM (Section 5.3.2.1: The margin)
    print("   Testing Linear SVM...")
    svm_linear = SVC(kernel='linear', random_state=42)
    svm_linear.fit(X_train, y_train)
    linear_pred = svm_linear.predict(X_test)
    linear_acc = accuracy_score(y_test, linear_pred)
    results['Linear'] = {'accuracy': linear_acc, 'model': svm_linear}
    print(f"     Linear SVM accuracy: {linear_acc:.4f}")
    
    # RBF SVM (Section 5.3.2.6: Kernels)
    print("   Testing RBF SVM...")
    svm_rbf = SVC(kernel='rbf', random_state=42)
    svm_rbf.fit(X_train, y_train)
    rbf_pred = svm_rbf.predict(X_test)
    rbf_acc = accuracy_score(y_test, rbf_pred)
    results['RBF'] = {'accuracy': rbf_acc, 'model': svm_rbf}
    print(f"     RBF SVM accuracy: {rbf_acc:.4f}")
    
    # Polynomial SVM
    print("   Testing Polynomial SVM...")
    svm_poly = SVC(kernel='poly', degree=2, random_state=42)
    svm_poly.fit(X_train, y_train)
    poly_pred = svm_poly.predict(X_test)
    poly_acc = accuracy_score(y_test, poly_pred)
    results['Polynomial'] = {'accuracy': poly_acc, 'model': svm_poly}
    print(f"     Polynomial SVM accuracy: {poly_acc:.4f}")
    
    return results

# Test SVM kernels on circles dataset (non-linearly separable)
print("5. Testing SVM with Different Kernels...")
svm_results = compare_svm_kernels(X_circles_train, X_circles_test, 
                                  y_circles_train, y_circles_test)

# Find best kernel
best_kernel = max(svm_results.keys(), key=lambda k: svm_results[k]['accuracy'])
print(f"   Best kernel: {best_kernel} (accuracy: {svm_results[best_kernel]['accuracy']:.4f})\n")

# Demonstrate support vectors (Section 5.3.2.4)
print("   Support Vector Analysis:")
best_svm = svm_results[best_kernel]['model']
n_support_vectors = best_svm.n_support_
total_support_vectors = best_svm.n_support_.sum()
total_samples = len(X_circles_train)

print(f"   Support vectors per class: {n_support_vectors}")
print(f"   Total support vectors: {total_support_vectors}/{total_samples} ({100*total_support_vectors/total_samples:.1f}%)")

# Demonstrate soft margin effect (Section 5.3.2.5)
print("\n   Soft Margin Effect (parameter C):")
C_values = [0.01, 0.1, 1, 10, 100]
for C in C_values:
    svm_soft = SVC(kernel='rbf', C=C, random_state=42)
    svm_soft.fit(X_circles_train, y_circles_train)
    soft_pred = svm_soft.predict(X_circles_test)
    soft_acc = accuracy_score(y_circles_test, soft_pred)
    n_sv = svm_soft.n_support_.sum()
    print(f"   C={C}: Accuracy={soft_acc:.4f}, Support vectors={n_sv}")

print()


### Example 5: Performance Evaluation Metrics

**Implementation:** Comprehensive evaluation using all metrics from Section 5.4.


In [ ]:
def comprehensive_evaluation(y_true, y_pred, y_proba=None, model_name=""):
    """
    Comprehensive evaluation using all metrics from Section 5.4
    """
    print(f"=== {model_name} Performance Evaluation ===\n")
    
    # 1. Accuracy (0/1 loss) - Section 5.4.1
    accuracy = accuracy_score(y_true, y_pred)
    error_rate = 1 - accuracy
    print(f"1. Accuracy (0/1 Loss):")
    print(f"   Accuracy: {accuracy:.4f}")
    print(f"   Error rate: {error_rate:.4f}\n")
    
    # 2. Sensitivity and Specificity - Section 5.4.2
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    sensitivity = tp / (tp + fn)  # Recall
    specificity = tn / (tn + fp)
    
    print(f"2. Sensitivity and Specificity:")
    print(f"   True Positives (TP): {tp}")
    print(f"   True Negatives (TN): {tn}")
    print(f"   False Positives (FP): {fp}")
    print(f"   False Negatives (FN): {fn}")
    print(f"   Sensitivity (Recall): {sensitivity:.4f}")
    print(f"   Specificity: {specificity:.4f}\n")
    
    # 3. ROC and AUC - Section 5.4.3
    if y_proba is not None:
        auc = roc_auc_score(y_true, y_proba[:, 1])
        fpr, tpr, thresholds = roc_curve(y_true, y_proba[:, 1])
        
        print(f"3. ROC Analysis:")
        print(f"   AUC (Area Under Curve): {auc:.4f}")
        print(f"   ROC curve points: {len(fpr)} threshold values\n")
    
    # 4. Confusion Matrix - Section 5.4.4
    cm = confusion_matrix(y_true, y_pred)
    print(f"4. Confusion Matrix:")
    print(f"   {cm}")
    
    # Calculate additional metrics
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    f1_score = 2 * (precision * sensitivity) / (precision + sensitivity) if (precision + sensitivity) > 0 else 0
    
    print(f"\nAdditional Metrics:")
    print(f"   Precision: {precision:.4f}")
    print(f"   F1-Score: {f1_score:.4f}")
    print("\n" + "="*50 + "\n")

# Test on high-dimensional dataset with multiple algorithms
print("6. Comprehensive Performance Evaluation...\n")

# Train multiple models on high-dimensional data
models = {
    'Naive Bayes': GaussianNB(),
    'Logistic Regression': LogisticRegression(random_state=42, max_iter=1000),
    'KNN (K=5)': KNeighborsClassifier(n_neighbors=5),
    'SVM (RBF)': SVC(kernel='rbf', probability=True, random_state=42)
}

results = {}
for name, model in models.items():
    print(f"Training {name}...")
    model.fit(X_highdim_train, y_highdim_train)
    y_pred = model.predict(X_highdim_test)
    
    # Get probabilities if available
    if hasattr(model, 'predict_proba'):
        y_proba = model.predict_proba(X_highdim_test)
    else:
        y_proba = None
    
    results[name] = {'predictions': y_pred, 'probabilities': y_proba}
    
    # Comprehensive evaluation
    comprehensive_evaluation(y_highdim_test, y_pred, y_proba, name)


### Example 6: Visualization of Decision Boundaries

**Visual demonstration** of how different algorithms create different decision boundaries.


In [ ]:
def plot_decision_boundaries(X, y, models, titles, figsize=(15, 10)):
    """
    Plot decision boundaries for different classification algorithms
    """
    # Create a mesh
    h = 0.02  # step size in the mesh
    x_min, x_max = X[:, 0].min() - 1, X[:, 0].max() + 1
    y_min, y_max = X[:, 1].min() - 1, X[:, 1].max() + 1
    xx, yy = np.meshgrid(np.arange(x_min, x_max, h),
                         np.arange(y_min, y_max, h))
    
    # Create subplots
    fig, axes = plt.subplots(2, 2, figsize=figsize)
    axes = axes.ravel()
    
    for i, (model, title) in enumerate(zip(models, titles)):
        # Train model
        model.fit(X, y)
        
        # Make predictions on mesh
        if hasattr(model, 'decision_function'):
            Z = model.decision_function(np.c_[xx.ravel(), yy.ravel()])
            Z = Z.reshape(xx.shape)
            levels = [0]
        else:
            Z = model.predict(np.c_[xx.ravel(), yy.ravel()])
            Z = Z.reshape(xx.shape)
            levels = None
        
        # Plot decision boundary
        ax = axes[i]
        ax.contourf(xx, yy, Z, levels=levels, alpha=0.3, cmap=plt.cm.RdYlBu)
        ax.contour(xx, yy, Z, levels=levels, colors='black', linestyles='--', linewidths=2)
        
        # Plot training points
        scatter = ax.scatter(X[:, 0], X[:, 1], c=y, cmap=plt.cm.RdYlBu, edgecolors='black', s=50)
        ax.set_title(title, fontsize=12, fontweight='bold')
        ax.set_xlim(xx.min(), xx.max())
        ax.set_ylim(yy.min(), yy.max())
        
        # Add accuracy score
        train_acc = model.score(X, y)
        ax.text(0.05, 0.95, f'Train Acc: {train_acc:.3f}', 
                transform=ax.transAxes, bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))
    
    plt.tight_layout()
    plt.suptitle('Decision Boundaries of Different Classification Algorithms', 
                 fontsize=16, fontweight='bold', y=1.02)
    plt.show()

# Create models for visualization
visualization_models = [
    GaussianNB(),
    LogisticRegression(random_state=42, max_iter=1000),
    KNeighborsClassifier(n_neighbors=5),
    SVC(kernel='rbf', random_state=42)
]

visualization_titles = [
    'Gaussian Naive Bayes',
    'Logistic Regression',
    'K-Nearest Neighbors (K=5)',
    'Support Vector Machine (RBF)'
]

print("7. Creating Decision Boundary Visualizations...")
print("   (Visualization will be displayed below)")

# Plot decision boundaries on simple dataset
plot_decision_boundaries(X_simple, y_simple, visualization_models, visualization_titles)


### Example 7: ROC Curves Comparison

**Implementation:** ROC curves for multiple algorithms to demonstrate Section 5.4.3 concepts.


In [ ]:
def plot_roc_curves(models, X_train, X_test, y_train, y_test):
    """
    Plot ROC curves for multiple models following Section 5.4.3
    """
    plt.figure(figsize=(10, 8))
    
    colors = ['blue', 'red', 'green', 'orange', 'purple']
    
    for i, (name, model) in enumerate(models.items()):
        # Train model
        model.fit(X_train, y_train)
        
        # Get probabilities
        if hasattr(model, 'predict_proba'):
            y_proba = model.predict_proba(X_test)[:, 1]
        else:
            # For models without predict_proba, use decision_function
            y_proba = model.decision_function(X_test)
        
        # Calculate ROC curve
        fpr, tpr, _ = roc_curve(y_test, y_proba)
        auc = roc_auc_score(y_test, y_proba)
        
        # Plot ROC curve
        plt.plot(fpr, tpr, color=colors[i % len(colors)], lw=2,
                label=f'{name} (AUC = {auc:.3f})')
    
    # Plot diagonal line (random classifier)
    plt.plot([0, 1], [0, 1], color='black', lw=2, linestyle='--', 
             label='Random Classifier (AUC = 0.500)')
    
    # Formatting
    plt.xlim([0.0, 1.0])
    plt.ylim([0.0, 1.05])
    plt.xlabel('False Positive Rate (1 - Specificity)', fontsize=12)
    plt.ylabel('True Positive Rate (Sensitivity)', fontsize=12)
    plt.title('ROC Curves Comparison\n(Section 5.4.3: The Area Under the ROC Curve)', 
              fontsize=14, fontweight='bold')
    plt.legend(loc="lower right", fontsize=10)
    plt.grid(True, alpha=0.3)
    
    # Add perfect classifier reference
    plt.plot([0, 0, 1], [0, 1, 1], color='gray', lw=1, linestyle=':', alpha=0.7,
             label='Perfect Classifier (AUC = 1.000)')
    
    plt.tight_layout()
    plt.show()

# Create models that support probability predictions
roc_models = {
    'Naive Bayes': GaussianNB(),
    'Logistic Regression': LogisticRegression(random_state=42, max_iter=1000),
    'SVM (RBF)': SVC(kernel='rbf', probability=True, random_state=42)
}

print("8. Creating ROC Curves Comparison...")
print("   (ROC curves will be displayed below)")

# Plot ROC curves
plot_roc_curves(roc_models, X_highdim_train, X_highdim_test, 
                y_highdim_train, y_highdim_test)


## Summary of Coding Examples

This section provided comprehensive implementations of all four main classification algorithms:

### **Algorithm Implementations:**
1. **Manual Gaussian Naive Bayes** - Complete implementation following Section 5.2.1.5
2. **Logistic Regression with Feature Expansion** - Demonstrating non-linear boundaries from Section 5.2.2.2
3. **KNN with Cross-Validation** - Optimal K selection following Section 5.3.1.1
4. **SVM with Multiple Kernels** - Linear, RBF, and Polynomial kernels from Section 5.3.2

### **Evaluation Methods:**
5. **Comprehensive Performance Metrics** - All metrics from Section 5.4
6. **Decision Boundary Visualization** - Visual comparison of algorithm behaviors
7. **ROC Curves Comparison** - Threshold-independent evaluation from Section 5.4.3

### **Key Learning Points:**
- **Mathematical foundations** translated into working code
- **Parameter tuning** techniques (K selection, C parameter, kernel choice)
- **Performance evaluation** using multiple metrics
- **Visualization** of algorithm behaviors and decision boundaries
- **Practical considerations** for real-world applications

These examples provide a complete toolkit for implementing and evaluating classification algorithms in practice.


## Decision Trees: Complete Guide

**Source:** Uri Almog - "Decision Trees, Explained" (May 8, 2022)

Decision trees are preferred for many applications due to their high explainability, simplicity to set up and train, and fast prediction times. They naturally work with tabular data and currently outperform neural networks on this type of data.


### How Decision Trees Work

**Prediction Process:**
The prediction process in a tree consists of a sequence of comparisons of the sample's attributes (features) with pre-learned threshold values. Starting from the top (root) and going downward (toward the leaves), each comparison determines if the sample goes left or right in the tree, determining the next comparison step. When the sample reaches a leaf (end node), the prediction is made based on the majority class in that leaf.

**Key Advantages:**
- **High explainability** - Easy to interpret and understand
- **No input normalization** required (unlike neural networks)
- **Fast training and prediction** times
- **Natural for tabular data** - Currently outperform neural networks on structured data
- **Can handle missing values** (though imputation is now preferred)

**Common Use Cases:**
- Recommendation systems
- Search engines
- Medical diagnosis
- Credit scoring


In [ ]:
# Decision Trees: Complete Implementation
print("=== Decision Trees: From Theory to Practice ===\n")

# Import additional libraries for decision trees
from sklearn.tree import DecisionTreeClassifier, export_graphviz
from sklearn.tree import plot_tree
from sklearn.datasets import load_iris
from sklearn.preprocessing import OneHotEncoder
import graphviz

# Load and prepare the Iris dataset (modified for demonstration)
print("1. Loading and preparing Iris dataset...")
iris = load_iris()
X = iris['data']
y = iris['target']
names = iris['target_names']
feature_names = iris['feature_names']

# One hot encoding for multi-class visualization
enc = OneHotEncoder()
Y = enc.fit_transform(y[:, np.newaxis]).toarray()

# Modify the dataset to make it more interesting (mix classes 1 and 2)
X[y==1, 2] = X[y==1, 2] + 0.3  # Add noise to petal length for class 1

# Split the dataset
X_train, X_test, Y_train, Y_test = train_test_split(
    X, Y, test_size=0.5, random_state=2
)

# Decrease train set to make classes more mixed
X_train = X_train[30:, :]
Y_train = Y_train[30:, :]

print(f"   Training set shape: {X_train.shape}")
print(f"   Test set shape: {X_test.shape}")
print(f"   Classes: {names}")
print(f"   Features: {feature_names}\n")


In [ ]:
# Visualize the modified dataset
print("2. Visualizing the modified Iris dataset...")

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Plot 1: Sepal length vs Sepal width
for target, target_name in enumerate(names):
    X_plot = X[y == target]
    axes[0].plot(X_plot[:, 0], X_plot[:, 1], linestyle='none', marker='o', 
                label=target_name, markersize=8)
axes[0].set_xlabel(feature_names[0])
axes[0].set_ylabel(feature_names[1])
axes[0].set_title('Full Dataset: Sepal Dimensions')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Plot 2: Petal length vs Petal width
for target, target_name in enumerate(names):
    X_plot = X[y == target]
    axes[1].plot(X_plot[:, 2], X_plot[:, 3], linestyle='none', marker='o', 
                label=target_name, markersize=8)
axes[1].set_xlabel(feature_names[2])
axes[1].set_ylabel(feature_names[3])
axes[1].set_title('Full Dataset: Petal Dimensions')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Visualize just the training set
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Plot 1: Training set - Sepal dimensions
for target, target_name in enumerate(names):
    X_plot = X_train[Y_train[:, target] == 1]
    if len(X_plot) > 0:
        axes[0].plot(X_plot[:, 0], X_plot[:, 1], linestyle='none', marker='o', 
                    label=target_name, markersize=8)
axes[0].set_xlabel(feature_names[0])
axes[0].set_ylabel(feature_names[1])
axes[0].set_title('Training Set: Sepal Dimensions')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Plot 2: Training set - Petal dimensions
for target, target_name in enumerate(names):
    X_plot = X_train[Y_train[:, target] == 1]
    if len(X_plot) > 0:
        axes[1].plot(X_plot[:, 2], X_plot[:, 3], linestyle='none', marker='o', 
                    label=target_name, markersize=8)
axes[1].set_xlabel(feature_names[2])
axes[1].set_ylabel(feature_names[3])
axes[1].set_title('Training Set: Petal Dimensions')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()
print()


In [ ]:
# Train and visualize the decision tree
print("3. Training Decision Tree and Creating Visualization...")

# Create and train the decision tree
iristree = DecisionTreeClassifier(max_depth=3, criterion='gini', random_state=0)
iristree.fit(X_train, enc.inverse_transform(Y_train))

# Create tree visualization using sklearn's plot_tree
plt.figure(figsize=(20, 10))
plot_tree(iristree, 
          feature_names=feature_names,
          class_names=names,
          filled=True,
          rounded=True,
          fontsize=10)
plt.title('Decision Tree Visualization', fontsize=16, fontweight='bold')
plt.show()

# Alternative: Create graphviz visualization (if available)
try:
    dot_data = export_graphviz(iristree, out_file=None, 
                              feature_names=feature_names,  
                              class_names=names,
                              filled=True, rounded=True,  
                              special_characters=True)  
    graph = graphviz.Source(dot_data)
    print("   Graphviz tree visualization created (display if environment supports it)")
except Exception as e:
    print(f"   Graphviz not available: {e}")

print("\n   Tree Analysis:")
print(f"   - Root node: {feature_names[iristree.tree_.feature[0]]} <= {iristree.tree_.threshold[0]:.2f}")
print(f"   - Tree depth: {iristree.get_depth()}")
print(f"   - Number of leaves: {iristree.get_n_leaves()}")
print(f"   - Criterion used: {iristree.criterion}")
print()


In [ ]:
# Evaluate the decision tree performance
print("4. Evaluating Decision Tree Performance...")

# Make predictions
iristrainpred = iristree.predict(X_train)
iristestpred = iristree.predict(X_test)

# Calculate precision for each class
from sklearn.metrics import precision_score, classification_report

print("   Training Set Precision (per class):")
for i, class_name in enumerate(names):
    train_precision = precision_score(enc.inverse_transform(Y_train), iristrainpred, 
                                     labels=[i], average='micro')
    print(f"   {class_name}: {train_precision:.4f}")

print("\n   Test Set Precision (per class):")
for i, class_name in enumerate(names):
    test_precision = precision_score(enc.inverse_transform(Y_test), iristestpred, 
                                    labels=[i], average='micro')
    print(f"   {class_name}: {test_precision:.4f}")

# Overall accuracy
train_acc = accuracy_score(enc.inverse_transform(Y_train), iristrainpred)
test_acc = accuracy_score(enc.inverse_transform(Y_test), iristestpred)

print(f"\n   Overall Accuracy:")
print(f"   Training: {train_acc:.4f}")
print(f"   Test: {test_acc:.4f}")

# Detailed classification report
print(f"\n   Detailed Classification Report:")
print(classification_report(enc.inverse_transform(Y_test), iristestpred, target_names=names))
print()


### Decision Tree Training Theory

**How Decision Trees Choose Splits:**

Decision trees use optimization criteria to choose the best feature and threshold for each split. The two most common criteria are:

#### 1. Gini Impurity

**Formula:** Gini = 1 - Σ(p_i)²

Where p_i is the probability of class i in the node.

**Interpretation:**
- **High Gini** = Heterogeneous population (mixed classes)
- **Low Gini** = Homogeneous population (mostly one class)
- **Minimum Gini** = 0 (pure node with single class)
- **Maximum Gini** = 1 - 1/C (for C classes, when evenly distributed)

**Why Gini Works:**
The Gini impurity is the expectation value of wrong classifications if classification is done randomly. The probability of randomly picking a sample from class i is p_i, and the probability of predicting the wrong class is (1-p_i). Summing p_i(1-p_i) over all classes gives the Gini formula.

#### 2. Entropy

**Formula:** Entropy = -Σ(p_i × log₂(p_i))

**Interpretation:**
- Measures the average number of yes/no questions needed to identify a sample's class
- **High entropy** = Many questions needed (mixed classes)
- **Low entropy** = Few questions needed (homogeneous population)
- **Minimum entropy** = 0 (pure node)

#### 3. Information Gain

**Formula:** Information Gain = Entropy(parent) - Weighted Average Entropy(children)

The tree chooses the split that maximizes information gain (or minimizes weighted average Gini/entropy).


In [ ]:
# Demonstrate Gini and Entropy calculations
print("5. Demonstrating Gini and Entropy Calculations...")

def calculate_gini(y):
    """Calculate Gini impurity for a node"""
    if len(y) == 0:
        return 0
    _, counts = np.unique(y, return_counts=True)
    probabilities = counts / len(y)
    gini = 1 - np.sum(probabilities**2)
    return gini

def calculate_entropy(y):
    """Calculate entropy for a node"""
    if len(y) == 0:
        return 0
    _, counts = np.unique(y, return_counts=True)
    probabilities = counts / len(y)
    entropy = -np.sum(probabilities * np.log2(probabilities + 1e-10))  # Add small epsilon to avoid log(0)
    return entropy

# Example calculations on our training data
print("   Example Gini and Entropy calculations:")

# Root node (all training data)
root_labels = enc.inverse_transform(Y_train).flatten()
root_gini = calculate_gini(root_labels)
root_entropy = calculate_entropy(root_labels)

print(f"   Root node:")
print(f"     Samples: {len(root_labels)}")
print(f"     Class distribution: {np.bincount(root_labels)}")
print(f"     Gini impurity: {root_gini:.4f}")
print(f"     Entropy: {root_entropy:.4f}")

# Calculate for different hypothetical splits
print(f"\n   Hypothetical splits:")

# Split 1: Petal width <= 0.8
split1_left = root_labels[X_train[:, 3] <= 0.8]
split1_right = root_labels[X_train[:, 3] > 0.8]

if len(split1_left) > 0:
    gini_left = calculate_gini(split1_left)
    entropy_left = calculate_entropy(split1_left)
    print(f"     Petal width <= 0.8: {len(split1_left)} samples, Gini={gini_left:.4f}, Entropy={entropy_left:.4f}")

if len(split1_right) > 0:
    gini_right = calculate_gini(split1_right)
    entropy_right = calculate_entropy(split1_right)
    print(f"     Petal width > 0.8: {len(split1_right)} samples, Gini={gini_right:.4f}, Entropy={entropy_right:.4f}")

# Weighted average Gini for this split
if len(split1_left) > 0 and len(split1_right) > 0:
    weighted_gini = (len(split1_left) * gini_left + len(split1_right) * gini_right) / len(root_labels)
    gini_reduction = root_gini - weighted_gini
    print(f"     Weighted average Gini: {weighted_gini:.4f}")
    print(f"     Gini reduction: {gini_reduction:.4f}")

print()

# Compare different criteria
print("6. Comparing Gini vs Entropy Criteria...")

# Train trees with different criteria
tree_gini = DecisionTreeClassifier(max_depth=3, criterion='gini', random_state=0)
tree_entropy = DecisionTreeClassifier(max_depth=3, criterion='entropy', random_state=0)

tree_gini.fit(X_train, enc.inverse_transform(Y_train))
tree_entropy.fit(X_train, enc.inverse_transform(Y_train))

# Compare performance
gini_train_acc = tree_gini.score(X_train, enc.inverse_transform(Y_train))
gini_test_acc = tree_gini.score(X_test, enc.inverse_transform(Y_test))

entropy_train_acc = tree_entropy.score(X_train, enc.inverse_transform(Y_train))
entropy_test_acc = tree_entropy.score(X_test, enc.inverse_transform(Y_test))

print(f"   Gini criterion:")
print(f"     Training accuracy: {gini_train_acc:.4f}")
print(f"     Test accuracy: {gini_test_acc:.4f}")

print(f"   Entropy criterion:")
print(f"     Training accuracy: {entropy_train_acc:.4f}")
print(f"     Test accuracy: {entropy_test_acc:.4f}")

print(f"   Difference in test accuracy: {abs(gini_test_acc - entropy_test_acc):.4f}")
print("   Note: The difference is typically small, as both criteria aim for similar goals.")
print()


### Decision Tree Parameters and Overfitting Prevention

**Key Parameters:**
- **max_depth**: Maximum depth of the tree (prevents overfitting)
- **min_samples_split**: Minimum samples required to split a node
- **min_samples_leaf**: Minimum samples required in a leaf node
- **criterion**: Split criterion ('gini' or 'entropy')
- **random_state**: For reproducible results

**Overfitting Prevention:**
1. **Limit tree depth** - Prevents overly complex trees
2. **Minimum samples per leaf** - Ensures leaves have enough data
3. **Minimum samples to split** - Prevents splits on very small nodes
4. **Pruning** - Remove branches that don't improve generalization

**When Training Stops:**
- Maximum depth reached
- All samples in a node belong to the same class (Gini/entropy = 0)
- Minimum samples threshold reached
- No improvement in split quality


In [ ]:
# Demonstrate parameter effects and overfitting prevention
print("7. Demonstrating Parameter Effects and Overfitting Prevention...")

# Create trees with different parameters
trees = {
    'Shallow (depth=2)': DecisionTreeClassifier(max_depth=2, random_state=0),
    'Deep (depth=10)': DecisionTreeClassifier(max_depth=10, random_state=0),
    'Min samples leaf=5': DecisionTreeClassifier(min_samples_leaf=5, random_state=0),
    'Min samples split=10': DecisionTreeClassifier(min_samples_split=10, random_state=0)
}

results = {}
for name, tree in trees.items():
    tree.fit(X_train, enc.inverse_transform(Y_train))
    train_acc = tree.score(X_train, enc.inverse_transform(Y_train))
    test_acc = tree.score(X_test, enc.inverse_transform(Y_test))
    
    results[name] = {
        'train_acc': train_acc,
        'test_acc': test_acc,
        'overfitting': train_acc - test_acc,
        'depth': tree.get_depth(),
        'leaves': tree.get_n_leaves()
    }

print("   Parameter Effects Analysis:")
print(f"   {'Tree Type':<20} {'Train Acc':<10} {'Test Acc':<10} {'Overfitting':<12} {'Depth':<8} {'Leaves':<8}")
print("   " + "-" * 70)

for name, metrics in results.items():
    print(f"   {name:<20} {metrics['train_acc']:<10.4f} {metrics['test_acc']:<10.4f} "
          f"{metrics['overfitting']:<12.4f} {metrics['depth']:<8} {metrics['leaves']:<8}")

print("\n   Key Observations:")
print("   - Shallow trees: Lower training accuracy but better generalization")
print("   - Deep trees: Higher training accuracy but potential overfitting")
print("   - Min samples constraints: Help prevent overfitting")
print()

# Feature importance analysis
print("8. Feature Importance Analysis...")
feature_importance = iristree.feature_importances_
feature_names_short = ['Sepal L', 'Sepal W', 'Petal L', 'Petal W']

print("   Feature importance scores:")
for name, importance in zip(feature_names_short, feature_importance):
    print(f"   {name}: {importance:.4f}")

# Visualize feature importance
plt.figure(figsize=(10, 6))
bars = plt.bar(feature_names_short, feature_importance, color='skyblue', edgecolor='navy')
plt.title('Feature Importance in Decision Tree', fontsize=14, fontweight='bold')
plt.xlabel('Features', fontsize=12)
plt.ylabel('Importance Score', fontsize=12)
plt.xticks(rotation=45)

# Add value labels on bars
for bar, importance in zip(bars, feature_importance):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
             f'{importance:.3f}', ha='center', va='bottom')

plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("   Note: Feature importance is calculated as the total reduction in impurity")
print("   contributed by each feature across all splits in the tree.")
print()


### Regression Trees

Decision trees can also be used for regression problems where the target variable is continuous rather than discrete.

**Key Differences for Regression:**
- **Target values**: Continuous instead of discrete classes
- **Leaf predictions**: Mean value of samples in each leaf instead of majority class
- **Split criteria**: Mean squared error (MSE) or mean absolute error (MAE) instead of Gini/entropy
- **Objective**: Minimize prediction error instead of maximizing class separation

**Regression Tree Process:**
1. **Split selection**: Choose feature and threshold that minimize MSE in resulting nodes
2. **Leaf prediction**: Use mean of target values in each leaf
3. **Stopping criteria**: Similar to classification (max depth, min samples, etc.)

**Advantages for Regression:**
- Handles non-linear relationships naturally
- No need for feature scaling
- Easy to interpret and visualize
- Can handle mixed data types


In [ ]:
# Regression Tree Example
print("9. Regression Tree Example...")

from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_squared_error, r2_score

# Create a synthetic regression dataset
np.random.seed(42)
X_reg = np.random.randn(200, 2)
y_reg = 2 * X_reg[:, 0] + 3 * X_reg[:, 1] + 0.5 * X_reg[:, 0] * X_reg[:, 1] + 0.1 * np.random.randn(200)

# Split the regression data
X_reg_train, X_reg_test, y_reg_train, y_reg_test = train_test_split(
    X_reg, y_reg, test_size=0.3, random_state=42
)

# Train regression tree
reg_tree = DecisionTreeRegressor(max_depth=4, random_state=42)
reg_tree.fit(X_reg_train, y_reg_train)

# Make predictions
y_reg_train_pred = reg_tree.predict(X_reg_train)
y_reg_test_pred = reg_tree.predict(X_reg_test)

# Evaluate performance
train_mse = mean_squared_error(y_reg_train, y_reg_train_pred)
test_mse = mean_squared_error(y_reg_test, y_reg_test_pred)
train_r2 = r2_score(y_reg_train, y_reg_train_pred)
test_r2 = r2_score(y_reg_test, y_reg_test_pred)

print(f"   Regression Tree Performance:")
print(f"   Training MSE: {train_mse:.4f}")
print(f"   Test MSE: {test_mse:.4f}")
print(f"   Training R²: {train_r2:.4f}")
print(f"   Test R²: {test_r2:.4f}")
print(f"   Tree depth: {reg_tree.get_depth()}")
print(f"   Number of leaves: {reg_tree.get_n_leaves()}")

# Visualize regression tree
plt.figure(figsize=(15, 8))
plot_tree(reg_tree, filled=True, rounded=True, fontsize=10)
plt.title('Regression Tree Visualization', fontsize=14, fontweight='bold')
plt.show()

print()


## Decision Trees Summary

### **Key Advantages:**
- **High interpretability** - Easy to understand and explain decisions
- **No data preprocessing** - Works with raw data, no normalization needed
- **Handles mixed data types** - Categorical and numerical features
- **Fast training and prediction** - Efficient algorithms
- **Natural feature selection** - Built-in importance ranking
- **Robust to outliers** - Less sensitive than parametric methods

### **Key Limitations:**
- **Overfitting tendency** - Can create overly complex trees
- **High variance** - Small data changes can lead to very different trees
- **Poor extrapolation** - Doesn't predict well outside training range
- **Greedy optimization** - Local optimal splits, not global optimum
- **Instability** - Sensitive to small changes in training data

### **When to Use Decision Trees:**
✅ **Good for:**
- Interpretability is important (medical diagnosis, loan approval)
- Mixed data types (categorical + numerical)
- Non-linear relationships
- Feature importance analysis
- Quick prototyping and baseline models

❌ **Avoid when:**
- High accuracy is critical (use ensemble methods instead)
- Data has strong linear relationships (linear models may be better)
- Large datasets with many features (memory intensive)
- Extrapolation beyond training range is needed

### **Best Practices:**
1. **Use ensemble methods** (Random Forest, Gradient Boosting) for better performance
2. **Tune parameters** to prevent overfitting (max_depth, min_samples_leaf)
3. **Cross-validate** to assess generalization performance
4. **Analyze feature importance** for insights and feature selection
5. **Consider pruning** for smaller, more generalizable trees

Decision trees serve as excellent building blocks for more sophisticated ensemble methods while remaining valuable standalone tools for interpretable machine learning.


## Python Machine Learning: Advanced Implementation

**Source:** Based on "Python Machine Learning" by Wei-Meng Lee (Wiley & Sons)

This section provides advanced Python implementations and best practices for machine learning algorithms, complementing the theoretical foundations covered earlier.

### Key Python Libraries for Machine Learning

#### Core Libraries
- **NumPy**: Numerical computing foundation
- **Pandas**: Data manipulation and analysis
- **Scikit-learn**: Machine learning algorithms and tools
- **Matplotlib/Seaborn**: Data visualization
- **SciPy**: Scientific computing

#### Advanced Libraries
- **XGBoost**: Gradient boosting framework
- **LightGBM**: Fast gradient boosting
- **CatBoost**: Categorical feature handling
- **Optuna**: Hyperparameter optimization
- **MLflow**: Machine learning lifecycle management

### Best Practices for Python ML

#### 1. Data Preprocessing Pipeline
```python
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.compose import ColumnTransformer

# Create preprocessing pipeline
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numerical_features),
        ('cat', LabelEncoder(), categorical_features)
    ]
)

# Combine with model
pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier())
])
```

#### 2. Cross-Validation and Model Selection
```python
from sklearn.model_selection import cross_val_score, GridSearchCV

# Cross-validation
scores = cross_val_score(model, X, y, cv=5, scoring='accuracy')

# Grid search
param_grid = {'n_estimators': [100, 200, 300], 'max_depth': [10, 20, None]}
grid_search = GridSearchCV(RandomForestClassifier(), param_grid, cv=5)
```

#### 3. Model Evaluation and Metrics
```python
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

# Comprehensive evaluation
y_pred = model.predict(X_test)
print(classification_report(y_test, y_pred))
print(confusion_matrix(y_test, y_pred))
```

### Advanced Techniques

#### Ensemble Methods
- **Bagging**: Random Forest, Extra Trees
- **Boosting**: AdaBoost, Gradient Boosting, XGBoost
- **Stacking**: Meta-learning approaches

#### Hyperparameter Optimization
- **Grid Search**: Exhaustive search
- **Random Search**: Random sampling
- **Bayesian Optimization**: Smart search strategies
- **Optuna**: Advanced optimization framework

#### Model Interpretability
- **SHAP**: SHapley Additive exPlanations
- **LIME**: Local Interpretable Model-agnostic Explanations
- **Feature Importance**: Tree-based and permutation importance


In [ ]:
# Advanced Python Machine Learning Techniques
print("=== Advanced Python ML Techniques ===\n")

# Import advanced libraries
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, VotingClassifier
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.metrics import classification_report, accuracy_score
import numpy as np

print("1. Advanced Pipeline with Preprocessing...")

# Create a more sophisticated dataset
from sklearn.datasets import make_classification
X_advanced, y_advanced = make_classification(
    n_samples=1000, n_features=20, n_informative=15, n_redundant=5,
    n_classes=3, random_state=42
)

X_adv_train, X_adv_test, y_adv_train, y_adv_test = train_test_split(
    X_advanced, y_advanced, test_size=0.3, random_state=42
)

# Advanced preprocessing pipeline
preprocessor = StandardScaler()

# Create pipeline with preprocessing and model
advanced_pipeline = Pipeline([
    ('scaler', preprocessor),
    ('classifier', RandomForestClassifier(random_state=42))
])

# Train and evaluate
advanced_pipeline.fit(X_adv_train, y_adv_train)
advanced_pred = advanced_pipeline.predict(X_adv_test)
advanced_acc = accuracy_score(y_adv_test, advanced_pred)

print(f"   Advanced Pipeline Accuracy: {advanced_acc:.4f}")

# Feature importance from the pipeline
feature_importance = advanced_pipeline.named_steps['classifier'].feature_importances_
print(f"   Top 5 most important features: {np.argsort(feature_importance)[-5:]}")

print()


In [ ]:
print("2. Ensemble Methods - Voting Classifier...")

# Create individual classifiers
rf_clf = RandomForestClassifier(n_estimators=100, random_state=42)
gb_clf = GradientBoostingClassifier(n_estimators=100, random_state=42)
svm_clf = SVC(probability=True, random_state=42)

# Create voting classifier (ensemble)
voting_clf = VotingClassifier(
    estimators=[
        ('rf', rf_clf),
        ('gb', gb_clf),
        ('svm', svm_clf)
    ],
    voting='soft'  # Use predicted probabilities
)

# Train and evaluate ensemble
voting_clf.fit(X_adv_train, y_adv_train)
voting_pred = voting_clf.predict(X_adv_test)
voting_acc = accuracy_score(y_adv_test, voting_pred)

print(f"   Voting Classifier Accuracy: {voting_acc:.4f}")

# Compare individual models
individual_accuracies = {}
for name, clf in voting_clf.named_estimators_.items():
    pred = clf.predict(X_adv_test)
    acc = accuracy_score(y_adv_test, pred)
    individual_accuracies[name] = acc
    print(f"   {name.upper()} Accuracy: {acc:.4f}")

print(f"   Ensemble improvement: {voting_acc - max(individual_accuracies.values()):.4f}")

print()


In [ ]:
print("3. Hyperparameter Optimization - Grid Search...")

# Define parameter grid for Random Forest
param_grid = {
    'classifier__n_estimators': [50, 100, 200],
    'classifier__max_depth': [5, 10, 15, None],
    'classifier__min_samples_split': [2, 5, 10]
}

# Create GridSearchCV
grid_search = GridSearchCV(
    advanced_pipeline,
    param_grid,
    cv=3,  # 3-fold cross-validation
    scoring='accuracy',
    n_jobs=-1,  # Use all available cores
    verbose=1
)

# Fit grid search
print("   Performing grid search (this may take a moment)...")
grid_search.fit(X_adv_train, y_adv_train)

# Get best parameters and score
best_params = grid_search.best_params_
best_score = grid_search.best_score_
test_score = grid_search.score(X_adv_test, y_adv_test)

print(f"   Best parameters: {best_params}")
print(f"   Best CV score: {best_score:.4f}")
print(f"   Test score: {test_score:.4f}")

print()


In [ ]:
print("4. Model Comparison and Evaluation...")

# Create multiple models for comparison
models = {
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),
    'Gradient Boosting': GradientBoostingClassifier(n_estimators=100, random_state=42),
    'SVM': SVC(random_state=42),
    'Logistic Regression': LogisticRegression(random_state=42, max_iter=1000),
    'Decision Tree': DecisionTreeClassifier(random_state=42),
    'Naive Bayes': GaussianNB()
}

# Train and evaluate all models
results = {}
for name, model in models.items():
    # Scale features for models that need it
    if name in ['SVM', 'Logistic Regression']:
        scaler = StandardScaler()
        X_train_scaled = scaler.fit_transform(X_adv_train)
        X_test_scaled = scaler.transform(X_adv_test)
        
        model.fit(X_train_scaled, y_adv_train)
        pred = model.predict(X_test_scaled)
    else:
        model.fit(X_adv_train, y_adv_train)
        pred = model.predict(X_adv_test)
    
    acc = accuracy_score(y_adv_test, pred)
    results[name] = acc
    print(f"   {name}: {acc:.4f}")

# Find best model
best_model = max(results, key=results.get)
print(f"\n   Best performing model: {best_model} ({results[best_model]:.4f})")

# Performance comparison visualization
plt.figure(figsize=(12, 6))
model_names = list(results.keys())
accuracies = list(results.values())

bars = plt.bar(model_names, accuracies, color='lightblue', edgecolor='navy')
plt.title('Model Performance Comparison', fontsize=14, fontweight='bold')
plt.xlabel('Models', fontsize=12)
plt.ylabel('Accuracy', fontsize=12)
plt.xticks(rotation=45, ha='right')
plt.ylim(0, 1)

# Add value labels on bars
for bar, acc in zip(bars, accuracies):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
             f'{acc:.3f}', ha='center', va='bottom')

plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print()


## Python ML Best Practices Summary

### **Code Organization and Reproducibility**

#### 1. **Random Seeds and Reproducibility**
```python
import numpy as np
import random

# Set seeds for reproducibility
np.random.seed(42)
random.seed(42)
# For sklearn models: random_state=42 parameter
```

#### 2. **Data Validation and Preprocessing**
```python
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder

# Always split data first
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Fit preprocessing on training data only
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)  # Don't fit on test data!
```

#### 3. **Cross-Validation Strategy**
```python
from sklearn.model_selection import cross_val_score, StratifiedKFold

# Use stratified CV for imbalanced datasets
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scores = cross_val_score(model, X_train, y_train, cv=cv, scoring='accuracy')
```

### **Performance Optimization**

#### 1. **Memory and Speed Optimization**
- Use `n_jobs=-1` for parallel processing
- Consider using `RandomizedSearchCV` instead of `GridSearchCV` for large parameter spaces
- Use sparse matrices for text data
- Consider feature selection to reduce dimensionality

#### 2. **Model Selection Strategy**
1. **Start simple**: Baseline models (Logistic Regression, Naive Bayes)
2. **Try tree-based**: Random Forest, Gradient Boosting
3. **Advanced methods**: SVM, Neural Networks
4. **Ensemble methods**: Combine best performers

### **Production Considerations**

#### 1. **Model Persistence**
```python
import joblib

# Save model
joblib.dump(model, 'trained_model.pkl')

# Load model
model = joblib.load('trained_model.pkl')
```

#### 2. **Error Handling and Logging**
```python
import logging

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

try:
    model.fit(X_train, y_train)
    logger.info("Model training completed successfully")
except Exception as e:
    logger.error(f"Model training failed: {e}")
```

### **Advanced Libraries to Explore**

#### 1. **Gradient Boosting Libraries**
- **XGBoost**: Excellent performance, handles missing values
- **LightGBM**: Fast training, good for large datasets
- **CatBoost**: Handles categorical features automatically

#### 2. **Hyperparameter Optimization**
- **Optuna**: Modern optimization framework
- **Hyperopt**: Bayesian optimization
- **Scikit-optimize**: Gaussian process-based optimization

#### 3. **Model Interpretability**
- **SHAP**: Unified framework for model explanations
- **LIME**: Local interpretable explanations
- **ELI5**: Debug and explain machine learning models

This comprehensive Python ML section provides both theoretical foundations and practical implementation guidance for advanced machine learning workflows.


## K-Nearest Neighbors: Complete Implementation Guide

**Source:** "Python Machine Learning" by Wei-Meng Lee - Chapter 9

### What Is K-Nearest Neighbors?

KNN is a relatively simple supervised machine learning algorithm that works by comparing the query instance's distance to other training samples and selecting the K-nearest neighbors. It then takes the majority of these K-neighbor classes to be the prediction of the query instance.

**Key Concept:** When k = 3, the closest three neighbors determine the classification. If k = 5, the closest five neighbors determine the classification. The choice of K significantly affects the prediction.

### Mathematical Foundation

**Euclidean Distance Formula:**
Given two points P₁(x₁, y₁) and P₂(x₂, y₂), the distance between them is:

d = √[(x₂ - x₁)² + (y₂ - y₁)²]

For n-dimensional space:
d = √[∑ᵢ₌₁ⁿ (x₂ᵢ - x₁ᵢ)²]


In [ ]:
# K-Nearest Neighbors: Manual Implementation from the Book
print("=== K-Nearest Neighbors: Complete Implementation ===\n")

import pandas as pd
import numpy as np
import operator
import seaborn as sns
import matplotlib.pyplot as plt

# Create the dataset from the book
knn_data = pd.DataFrame({
    'x': [1, 2, 4, 3, 3, 5, 5],
    'y': [1, 2, 3, 3, 5, 6, 4],
    'c': ['A', 'A', 'B', 'A', 'B', 'B', 'B']
})

print("1. Dataset from the book:")
print(knn_data)
print()

# Visualize the dataset
plt.figure(figsize=(8, 6))
colors = ['red' if i == 'A' else 'blue' for i in knn_data['c']]
plt.scatter(knn_data['x'], knn_data['y'], c=colors, s=100, alpha=0.7)
plt.xlabel('X coordinate')
plt.ylabel('Y coordinate')
plt.title('KNN Dataset: Class A (Red) and Class B (Blue)')
plt.grid(True, alpha=0.3)
plt.xlim(0, 6)
plt.ylim(0, 7)

# Add labels for each point
for i, row in knn_data.iterrows():
    plt.annotate(f"({row['x']},{row['y']})", 
                (row['x'], row['y']), 
                xytext=(5, 5), textcoords='offset points', fontsize=9)

plt.show()

print("2. Implementing KNN from scratch (following the book):")

# Euclidean distance function from the book
def euclidean_distance(pt1, pt2, dimension):
    """Calculate Euclidean distance between two points"""
    distance = 0
    for x in range(dimension):
        distance += np.square(pt1.iloc[0, x] - pt2.iloc[x])
    return np.sqrt(distance)

# KNN implementation from the book
def knn(training_points, test_point, k):
    """KNN implementation following the book's approach"""
    distances = {}
    
    # The number of axes we are dealing with
    dimension = test_point.shape[1]
    
    # Calculating euclidean distance between each point in the training data and test data
    for x in range(len(training_points)):
        dist = euclidean_distance(test_point, training_points.iloc[x], dimension)
        # Record the distance for each training points
        distances[x] = dist
    
    # Sort the distances
    sorted_d = sorted(distances.items(), key=operator.itemgetter(1))
    
    # To store the neighbors
    neighbors = []
    
    # Extract the top k neighbors
    for x in range(k):
        neighbors.append(sorted_d[x][0])
    
    # For each neighbor found, find out its class
    class_counter = {}
    for x in range(len(neighbors)):
        # Find out the class for that particular point
        cls = training_points.iloc[neighbors[x]]['c']
        if cls in class_counter:
            class_counter[cls] += 1
        else:
            class_counter[cls] = 1
    
    # Sort the class_counter in descending order
    sorted_counter = sorted(class_counter.items(),
                          key=operator.itemgetter(1),
                          reverse=True)
    
    # Return the class with the most count, as well as the neighbors found
    return(sorted_counter[0][0], neighbors)

print("   KNN implementation completed!")
print()


In [ ]:
print("3. Making predictions (following the book's example):")

# Test point from the book
test_set = [[3, 3.9]]
test = pd.DataFrame(test_set, columns=['x', 'y'])

print(f"   Test point: ({test_set[0][0]}, {test_set[0][1]})")

# Make prediction with k=5
cls, neighbors = knn(knn_data, test, 5)
print(f"   Predicted Class (k=5): {cls}")

# Show the neighbors used
print("   Neighbors used:")
for i, neighbor_idx in enumerate(neighbors):
    neighbor = knn_data.iloc[neighbor_idx]
    print(f"     Neighbor {i+1}: ({neighbor['x']}, {neighbor['y']}) - Class {neighbor['c']}")

print()

# Visualize the prediction
plt.figure(figsize=(10, 8))
colors = ['red' if i == 'A' else 'blue' for i in knn_data['c']]
plt.scatter(knn_data['x'], knn_data['y'], c=colors, s=100, alpha=0.7, label='Training points')

# Plot test point
plt.scatter(test_set[0][0], test_set[0][1], c='yellow', s=150, marker='s', 
           edgecolors='black', linewidth=2, label='Test point')

# Highlight the k=5 neighbors
neighbor_points = knn_data.iloc[neighbors]
plt.scatter(neighbor_points['x'], neighbor_points['y'], c='green', s=120, 
           marker='^', edgecolors='black', linewidth=2, label=f'k=5 neighbors')

# Draw circles to show distance
for i, neighbor_idx in enumerate(neighbors):
    neighbor = knn_data.iloc[neighbor_idx]
    dist = euclidean_distance(test, neighbor, 2)
    circle = plt.Circle((test_set[0][0], test_set[0][1]), dist, 
                       color='gray', alpha=0.2, linestyle='--')
    plt.gca().add_patch(circle)

plt.xlabel('X coordinate')
plt.ylabel('Y coordinate')
plt.title(f'KNN Prediction (k=5): Test point classified as Class {cls}')
plt.legend()
plt.grid(True, alpha=0.3)
plt.xlim(0, 6)
plt.ylim(0, 7)
plt.show()

print()


In [ ]:
print("4. Visualizing different values of K (following the book's approach):")

# Create visualization for different k values
fig, axes = plt.subplots(2, 2, figsize=(15, 12))
axes = axes.ravel()

k_values = [1, 3, 5, 7]
colors_base = ['red' if i == 'A' else 'blue' for i in knn_data['c']]

for idx, k in enumerate(k_values):
    ax = axes[idx]
    
    # Plot training points
    ax.scatter(knn_data['x'], knn_data['y'], c=colors_base, s=100, alpha=0.7)
    
    # Plot test point
    ax.scatter(test_set[0][0], test_set[0][1], c='yellow', s=150, 
              marker='s', edgecolors='black', linewidth=2)
    
    # Get prediction and neighbors for this k
    cls, neighbors = knn(knn_data, test, k)
    
    # Highlight neighbors
    neighbor_points = knn_data.iloc[neighbors]
    ax.scatter(neighbor_points['x'], neighbor_points['y'], c='green', s=120, 
              marker='^', edgecolors='black', linewidth=2)
    
    # Draw circle for the furthest neighbor
    furthest_neighbor = knn_data.iloc[neighbors[-1]]
    max_dist = euclidean_distance(test, furthest_neighbor, 2)
    
    circle_color = 'red' if cls == 'A' else 'blue'
    circle = plt.Circle((test_set[0][0], test_set[0][1]), max_dist, 
                       color=circle_color, alpha=0.3, linestyle='--')
    ax.add_patch(circle)
    
    ax.set_xlim(0, 6)
    ax.set_ylim(0, 7)
    ax.set_xlabel('X coordinate')
    ax.set_ylabel('Y coordinate')
    ax.set_title(f'k = {k}: Predicted Class = {cls}')
    ax.grid(True, alpha=0.3)
    
    # Print details
    print(f"   k = {k}: Class = {cls}")
    print(f"     Neighbors: {[knn_data.iloc[i]['c'] for i in neighbors]}")
    print(f"     Class distribution: {dict(zip(*np.unique([knn_data.iloc[i]['c'] for i in neighbors], return_counts=True)))}")

plt.tight_layout()
plt.show()

print()


### KNN Algorithm Analysis

**Key Insights from the Implementation:**

#### 1. **Effect of K Value**
- **k = 1**: Most sensitive to noise, high variance
- **k = 3**: Balanced approach, commonly used
- **k = 5**: More stable, reduces noise impact
- **k = 7**: Very stable but may miss local patterns

#### 2. **Distance Metrics**
The book focuses on Euclidean distance, but other options include:
- **Manhattan Distance**: |x₁ - x₂| + |y₁ - y₂|
- **Minkowski Distance**: Generalization of Euclidean and Manhattan
- **Cosine Similarity**: For high-dimensional data

#### 3. **KNN Characteristics**
**Advantages:**
- Simple to understand and implement
- No assumptions about data distribution
- Works well with non-linear decision boundaries
- Can handle multi-class problems naturally

**Disadvantages:**
- Computationally expensive for large datasets
- Sensitive to irrelevant features
- Requires feature scaling for different scales
- Memory intensive (stores all training data)

#### 4. **Best Practices**
1. **Feature Scaling**: Always scale features when using distance-based algorithms
2. **K Selection**: Use cross-validation to find optimal K
3. **Distance Metrics**: Choose appropriate distance metric for your data
4. **Dimensionality**: Consider dimensionality reduction for high-dimensional data

### Comparison with sklearn Implementation

The manual implementation above demonstrates the core concepts, while sklearn's `KNeighborsClassifier` provides:
- Optimized distance calculations
- Multiple distance metrics
- Weighted voting options
- Efficient algorithms for large datasets
